**Author:** Adebanji Oluwatimileyin Adelowo  
**GitHub:** [adebanjiadelowo](https://github.com/adebanjiadelowo)

<div align="center">
<h1>Enterprise NL2SQL Pipeline</h1>
<h2>Chinook — Real Database Execution</h2>
<p>End-to-end NL2SQL against a real 11-table SQLite music store database, with live query execution.</p>
</div>

<hr>

## Overview

The hand-crafted notebooks generate SQL but never execute it. This notebook uses the
**Chinook** database — a real SQLite music store with 11 tables and real data — to show
the full loop: natural language in, actual query results out.

Key additions over the hand-crafted project:

| Feature | Hand-crafted | This notebook |
|---|---|---|
| Schema source | Manually written | Auto-introspected from SQLite |
| Data | Sample rows only | Real rows |
| SQL execution | None | `sqlite3` — shows actual results |
| Table descriptions | Hand-written | Auto-generated from column names |

> **Prerequisite:** `OPENAI_API_KEY` in a `.env` file.
> The Chinook database is downloaded automatically in the next cell.

In [ ]:
!pip install -q openai python-dotenv pandas

In [ ]:
import os, re, json, sqlite3, urllib.request
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"


## 1 – Download the Chinook database

In [ ]:
DB_PATH = "chinook.db"

if not os.path.exists(DB_PATH):
    url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
    print("Downloading Chinook database ...")
    urllib.request.urlretrieve(url, DB_PATH)
    print(f"Saved to {DB_PATH}")
else:
    print(f"Using existing {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")


## 2 – Auto-introspect the schema

Instead of hand-writing `CREATE TABLE` blocks, we read them directly from the database.
This approach works for **any** SQLite database with zero manual effort.


In [ ]:
cursor = conn.cursor()

# All user tables (excluding SQLite internal tables)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;")
TABLE_NAMES = [row[0] for row in cursor.fetchall()]
print(f"Tables ({len(TABLE_NAMES)}):", TABLE_NAMES)


In [ ]:
def get_create_sql(table: str) -> str:
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name=?", (table,))
    row = cursor.fetchone()
    return row[0] if row else ""


def get_sample_rows(table: str, n: int = 3) -> str:
    df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT {n}", conn)
    return df.to_string(index=False)


def build_schema_block(table: str) -> str:
    return f"{get_create_sql(table)};\n/* {3} example rows\n{get_sample_rows(table)}\n*/"


SCHEMAS = {t: build_schema_block(t) for t in TABLE_NAMES}
print("Schema block for Employee:")
print(SCHEMAS.get("Employee", SCHEMAS.get("employee", "not found")))


## 3 – Auto-generate table descriptions

We build one-line descriptions from column names so the table selector has enough
context to route questions — without any manual writing.


In [ ]:
def describe_table(table: str) -> str:
    cursor.execute(f"PRAGMA table_info({table})")
    cols = [row[1] for row in cursor.fetchall()]
    cols_str = ", ".join(cols[:8])
    suffix = f" (+{len(cols)-8} more)" if len(cols) > 8 else ""
    return f"{table}: {cols_str}{suffix}"


CATALOGUE = {t: describe_table(t) for t in TABLE_NAMES}
print("Table catalogue:")
for t, d in CATALOGUE.items():
    print(f"  {d}")


## 4 – Pipeline functions

In [ ]:
SYSTEM_SELECT = (
    "You are a database assistant. "
    "Given a list of database tables and their descriptions, identify which tables are required "
    "to answer the user's question. "
    "Respond ONLY with a valid JSON array of table name strings. "
    "Do not include any explanation or markdown fencing."
)

USER_SELECT = """
### Available tables
{tables}

### Question
{question}

Which tables are needed? Reply with a JSON array of table names only.
"""


def select_tables(question: str) -> list[str]:
    tables_text = "\n".join(f"{t}: {d}" for t, d in CATALOGUE.items())
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_SELECT},
            {"role": "user",   "content": USER_SELECT.format(tables=tables_text, question=question)},
        ],
        temperature=0,
    )
    selected = json.loads(response.choices[0].message.content.strip())
    known = {t.lower() for t in TABLE_NAMES}
    return [t for t in selected if t.lower() in known]


def strip_fencing(text: str) -> str:
    text = re.sub(r"^```(?:sql)?\s*", "", text.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s*```$", "", text.strip()).strip()


def generate_sql(question: str, selected: list[str]) -> str:
    schema = "\n\n".join(SCHEMAS[t] for t in selected if t in SCHEMAS)
    prompt = (
        schema + "\n\n"
        "-- Write valid SQLite. Use only the tables listed above.\n"
        "-- Return ONLY the raw SQL query, no explanation, no markdown fencing.\n\n"
        "Question: " + question
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an expert SQL engineer. Return only valid SQLite, no markdown."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0,
    )
    return strip_fencing(response.choices[0].message.content)


## 5 – Run against the real database

For each question we:
1. Select relevant tables
2. Generate SQL
3. Execute against Chinook and display real results


In [ ]:
TEST_QUESTIONS = [
    "How many customers are there in each country? Show top 5 by count.",
    "What are the top 5 best-selling tracks by total revenue?",
    "Which employee supports the most customers?",
    "List all albums by AC/DC.",
    "What is the total revenue per genre?",
    "Which 3 customers have spent the most in total?",
    "What is the average track length in minutes per genre?",
]

results = []
for q in TEST_QUESTIONS:
    selected = select_tables(q)
    sql      = generate_sql(q, selected)
    try:
        df_result = pd.read_sql_query(sql, conn)
        error = None
    except Exception as e:
        df_result = None
        error = str(e)
    results.append({"question": q, "tables": selected, "sql": sql,
                    "result": df_result, "error": error})
    status = "OK" if error is None else f"ERROR: {error}"
    print(f"[{status}] {q[:60]}")


## 6 – Results

In [ ]:
for r in results:
    print("=" * 70)
    print("Q      :", r["question"])
    print("Tables :", r["tables"])
    print("SQL    :")
    print(r["sql"])
    print()
    if r["result"] is not None:
        print(r["result"].to_string(index=False))
    else:
        print("ERROR:", r["error"])
    print()


## 7 – Prompt size reduction

In [ ]:
full_size = sum(len(SCHEMAS[t].split()) for t in TABLE_NAMES)
rows = []
for r in results:
    sel_size = sum(len(SCHEMAS[t].split()) for t in r["tables"] if t in SCHEMAS)
    reduction = round((1 - sel_size / full_size) * 100)
    rows.append({
        "Question": r["question"][:55],
        "Tables selected": len(r["tables"]),
        "Prompt words": sel_size,
        "Reduction vs full schema": f"{reduction}%",
        "Executed OK": r["error"] is None,
    })

summary = pd.DataFrame(rows)
print(f"Full schema size : ~{full_size} words ({len(TABLE_NAMES)} tables)")
print()
print(summary.to_string(index=False))


## Conclusions

Using the real Chinook database adds three things the hand-crafted notebooks cannot provide:

1. **Auto-introspected schema** — zero manual `CREATE TABLE` writing; works on any SQLite DB
2. **Real data** — sample rows come from actual records, not invented examples
3. **Live execution** — we verify the SQL actually runs and returns correct results

The pipeline structure (table selector → SQL generator) is identical to the hand-crafted version,
confirming it generalises across databases without code changes — only the catalogue and schema
blocks need to be rebuilt, which `describe_table()` and `build_schema_block()` do automatically.
